# Predict With Our Amazon Comprehend Custom Classifier Model

<img src="img/comprehend.png" width="80%" align="left">

## Note that Amazon Comprehend is currently only supported in a subset of regions: 

* US East (N. Virginia), US East (Ohio), US West (Oregon)
* Canada (Central)
* Europe (London), Europe (Ireland), Europe (Frankfurt)
* Asia Pacific (Mumbai), Asia Pacific (Seoul), Asia Pacific (Tokyo), Asia Pacific (Singapore), Asia Pacific (Sydney)

You can check https://aws.amazon.com/about-aws/global-infrastructure/regional-product-services/ for details and updates. 

In [2]:
import boto3
import sagemaker
import pandas as pd

sess = sagemaker.Session()
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name

from botocore.config import Config

config = Config(retries={"max_attempts": 10, "mode": "adaptive"})

comprehend = boto3.Session().client(service_name="comprehend", region_name=region)

In [1]:
#Help find where the %store command variables are stored
#found them at the following path: sagemaker-user@default:~/.ipython/profile_default/db/autorestore
import IPython
print(IPython.paths.get_ipython_dir())

/home/sagemaker-user/.ipython


In [3]:
%store -r comprehend_training_job_arn

In [4]:
try:
    comprehend_training_job_arn
except NameError:
    print("***************************************************************************")
    print("[ERROR] PLEASE WAIT FOR THE PREVIOUS NOTEBOOK TO FINISH *******************")
    print("[ERROR] OR THIS NOTEBOOK WILL NOT RUN PROPERLY ****************************")
    print("***************************************************************************")

In [5]:
print(comprehend_training_job_arn)

arn:aws:comprehend:us-east-1:891377026966:document-classifier/Amazon-Customer-Reviews-Classifier-1719701765


In [6]:
%store -r comprehend_endpoint_arn

In [7]:
try:
    comprehend_endpoint_arn
except NameError:
    print("***************************************************************************")
    print("[ERROR] PLEASE WAIT FOR THE PREVIOUS NOTEBOOK TO FINISH *******************")
    print("[ERROR] OR THIS NOTEBOOK WILL NOT RUN PROPERLY ****************************")
    print("***************************************************************************")

In [8]:
print(comprehend_endpoint_arn)

arn:aws:comprehend:us-east-1:891377026966:document-classifier-endpoint/comprehend-inference-ep-30-01-02-00


# Deploy Endpoint

In [9]:
describe_response = comprehend.describe_endpoint(EndpointArn=comprehend_endpoint_arn)
print(describe_response)

{'EndpointProperties': {'EndpointArn': 'arn:aws:comprehend:us-east-1:891377026966:document-classifier-endpoint/comprehend-inference-ep-30-01-02-00', 'Status': 'CREATING', 'ModelArn': 'arn:aws:comprehend:us-east-1:891377026966:document-classifier/Amazon-Customer-Reviews-Classifier-1719701765', 'DesiredInferenceUnits': 1, 'CurrentInferenceUnits': 0, 'CreationTime': datetime.datetime(2024, 6, 30, 1, 2, 0, 673000, tzinfo=tzlocal()), 'LastModifiedTime': datetime.datetime(2024, 6, 30, 1, 2, 0, 673000, tzinfo=tzlocal())}, 'ResponseMetadata': {'RequestId': '0ca6d678-bd2b-46b0-a83f-e174ead960b9', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '0ca6d678-bd2b-46b0-a83f-e174ead960b9', 'content-type': 'application/x-amz-json-1.1', 'content-length': '536', 'date': 'Sun, 30 Jun 2024 01:02:40 GMT'}, 'RetryAttempts': 0}}


# Check Endpoint Status

In [10]:
from IPython.core.display import display, HTML

display(
    HTML(
        '<b>Review <a target="blank" href="https://console.aws.amazon.com/comprehend/v2/home?region={}#classifier-details/{}/endpoints/{}/details">Comprehend Model Endpoint</a></b>'.format(
            region, comprehend_training_job_arn, comprehend_endpoint_arn
        )
    )
)

/tmp/ipykernel_2638/1528860281.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [11]:
import time

max_time = time.time() + 3 * 60 * 60  # 3 hours
while time.time() < max_time:
    describe_response = comprehend.describe_endpoint(EndpointArn=comprehend_endpoint_arn)
    status = describe_response["EndpointProperties"]["Status"]
    print("Endpoint: {}".format(status))

    if status == "IN_SERVICE" or status == "IN_ERROR":
        break

    time.sleep(5)

Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CREATING
Endpoint: CR

# [INFO] _Feel free to continue to the next workshop section while this notebook is running._

# Predict with Endpoint

In [12]:
txt = """I loved it!  I will recommend this to everyone."""

response = comprehend.classify_document(Text=txt, EndpointArn=comprehend_endpoint_arn)

import json

print(json.dumps(response, indent=2, default=str))

{
  "Classes": [
    {
      "Name": "5",
      "Score": 0.9269999861717224
    },
    {
      "Name": "4",
      "Score": 0.04780000075697899
    },
    {
      "Name": "1",
      "Score": 0.009600000455975533
    }
  ],
  "ResponseMetadata": {
    "RequestId": "960b4545-900e-413d-83be-c98221554e87",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "x-amzn-requestid": "960b4545-900e-413d-83be-c98221554e87",
      "content-type": "application/x-amz-json-1.1",
      "content-length": "136",
      "date": "Sun, 30 Jun 2024 01:11:15 GMT"
    },
    "RetryAttempts": 0
  }
}


In [13]:
txt = """It's OK."""

response = comprehend.classify_document(Text=txt, EndpointArn=comprehend_endpoint_arn)

import json

print(json.dumps(response, indent=2, default=str))

{
  "Classes": [
    {
      "Name": "3",
      "Score": 0.6840999722480774
    },
    {
      "Name": "4",
      "Score": 0.125900000333786
    },
    {
      "Name": "2",
      "Score": 0.11630000174045563
    }
  ],
  "ResponseMetadata": {
    "RequestId": "f035de15-eb14-4aa3-bf3d-c532f08f59e2",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "x-amzn-requestid": "f035de15-eb14-4aa3-bf3d-c532f08f59e2",
      "content-type": "application/x-amz-json-1.1",
      "content-length": "133",
      "date": "Sun, 30 Jun 2024 01:11:24 GMT"
    },
    "RetryAttempts": 0
  }
}


In [14]:
txt = """Really bad.  I hope they don't make this anymore."""

response = comprehend.classify_document(Text=txt, EndpointArn=comprehend_endpoint_arn)

import json

print(json.dumps(response, indent=2, default=str))

{
  "Classes": [
    {
      "Name": "1",
      "Score": 0.5054000020027161
    },
    {
      "Name": "3",
      "Score": 0.22769999504089355
    },
    {
      "Name": "2",
      "Score": 0.20960000157356262
    }
  ],
  "ResponseMetadata": {
    "RequestId": "0dbb0564-f8a9-489b-a849-bf8799060723",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "x-amzn-requestid": "0dbb0564-f8a9-489b-a849-bf8799060723",
      "content-type": "application/x-amz-json-1.1",
      "content-length": "135",
      "date": "Sun, 30 Jun 2024 01:11:28 GMT"
    },
    "RetryAttempts": 0
  }
}


# Release Resources

In [15]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>

In [ ]:
%%javascript

try {
    Jupyter.notebook.save_checkpoint();
    Jupyter.notebook.session.delete();
}
catch(err) {
    // NoOp
}